# Image-to-image generation

`diffusers` provides a dedicated pipeline for image-to-image generation: `StableDiffusionImg2ImgPipeline`. This allows us to take an existing image and text to create a modified image.

In [ ]:
import torch

In [ ]:
if torch.cuda.is_available():
    # Shows the nVidia GPUs, if this system has any
    !nvidia-smi

In [ ]:
!pip install diffusers==0.11.1
!pip install transformers scipy ftfy accelerate

In [ ]:
# This is added to get around some issues of Torch not loading models correctly (test on Mac OS X and Kubuntu Linux)
!pip install --upgrade huggingface-hub==0.26.2 transformers==4.46.1 tokenizers==0.20.1 diffusers==0.31.0

In [ ]:
from diffusers import StableDiffusionImg2ImgPipeline
# from diffusers import StableDiffusionXLImg2ImgPipeline

from PIL import Image

pipe_img2img = StableDiffusionImg2ImgPipeline.from_pretrained("CompVis/stable-diffusion-v1-4")
# pipe_img2img = StableDiffusioXLImg2ImgPipeline.from_pretrained("stabilityai/stable-diffusion-xl-refiner-1.0")


if torch.cuda.is_available():
    device=torch.device("cuda")
elif torch.backends.mps.is_available():
    device=torch.device("mps")

pipe_img2img = pipe_img2img.to(device)

In [ ]:
image_list = [
    # "/content/broccoli_01b_4.png"
    # "/broccoli_01b.jpg",
    "/content/broccoli_01b_2.png",
    "/content/broccoli_01b_4.png",
    "/content/broccoli_01b_8.png",
    "/content/broccoli_08s_2.png",
    "/content/broccoli_08s_4.png",
    "/content/broccoli_08s_8.png",
    "/content/broccoli_10s_2.png",
    "/content/broccoli_10s_4.png",
    "/content/broccoli_10s_8.png"
]


classification = "broccoli"
num_images = 1
strength_val = 0.8
guidance_scale = 7.5
generator = torch.Generator(device).manual_seed(1022)


for i in range(len(image_list)):



    init_image = Image.open(image_list[i])
    # Resize the image to 512x512, as Stable Diffusion models are trained on this size.
    # This is important for consistent results.
    init_image = init_image.resize((512, 512))
    init_image = [init_image] * num_images

    prompt_img2img = [f"A photograph of a {classification}, in a place you would normally find a {classification}"] * num_images

    image = pipe_img2img(
        prompt=prompt_img2img,
        image=init_image,
        strength=strength_val,
        guidance_scale=guidance_scale,
        num_inference_steps=70,
        generator=generator
    ).images[0]

    image.save(f"diffused_{i+1}.png")
